# Symbolic Knowledge Discovery with FCA, TD at IDMC 2026

This Jupyter notebook provides the first hands-on expirience with Formal Concept Analysis in Python.
It is designed in a way so that you can practice converting the math of FCA into Python code, so that you can evaluate the efficiency of the algorithms you've studied (e.g. NextClosure), and so that you can see how FCA output can look on the real data.

The notebook is split into 6 (+1) parts:

0. Load the data
1. Basic FCA operations
2. Mining concepts in a smart way (i.e. NextClosure),
3. Mining implications in a smart way (through Carpathia-G algorithm),
4. Mining association rules (with Luxenburger basis),
5. Trying out the real data (the Titanic dataset),
6. Pushing the concepts to GraphRAG (a small hint on the project).

But keep calm, *you are only asked to write some code in Parts 1, 2, and 3*: the other sections serve demonstration purposes only.

## Part 0. Load the data

In [1]:
!pip install --quiet caspailleur

Download one of the classical FCA datasets available at the Formal Context Repository: https://fcarepository.org/

In [2]:
import pandas as pd
from caspailleur.io import from_fca_repo
df, metadata = from_fca_repo('newzealand_en.cxt')
df

,Hiking,Observing Nature,Sightseeing Flights,Jet Boating,Wildwater Rafting,Bungee Jumping,Parachute Gliding,Skiing
Stewart Island,True,True,True,False,False,False,False,False
Fjordland NP,True,True,True,False,False,False,False,False
Invercargill,True,True,True,False,False,False,False,False
Milford Sound,True,True,True,False,False,False,False,False
MT. Aspiring NP,True,True,True,False,False,False,False,False
Te Anau,True,True,True,True,False,False,False,False
Dunedin,True,True,True,False,False,False,False,False
Oamaru,True,True,False,False,False,False,False,False
Queenstown,True,False,True,True,True,True,True,True
Wanaka,True,False,True,True,True,True,True,True


To make the code more structured and more similar to the mathematical definitions, we represent the dataset with this simple `FormalContext` class:

In [3]:
from typing import NamedTuple

class FormalContext(NamedTuple):
  objects: set[str]
  attributes: set[str]
  relations: set[tuple[str, str]]

In [4]:
def pandas_to_sets(df) -> FormalContext:
  objects = set(df.index)
  attributes = set(df.columns)
  relations = {(obj, attr) for obj in objects for attr in attributes if df.at[obj, attr]}
  return FormalContext(objects, attributes, relations)

In [5]:
context = pandas_to_sets(df)
print(f"{context.objects=}")
print(f"{context.attributes=}")
print(f"{context.relations=}")

context.objects={'MT. Aspiring NP', 'Dunedin', 'Stewart Island', 'Queenstown', 'Fjordland NP', 'Te Anau', 'Invercargill', 'Oamaru', 'Wanaka', 'Haast', 'Catlins', 'Otago Peninsula', 'Milford Sound'}
context.attributes={'Jet Boating', 'Parachute Gliding', 'Wildwater Rafting', 'Bungee Jumping', 'Observing Nature', 'Sightseeing Flights', 'Hiking', 'Skiing'}
context.relations={('Stewart Island', 'Hiking'), ('MT. Aspiring NP', 'Sightseeing Flights'), ('Haast', 'Hiking'), ('Milford Sound', 'Hiking'), ('Wanaka', 'Wildwater Rafting'), ('Otago Peninsula', 'Hiking'), ('Fjordland NP', 'Observing Nature'), ('Dunedin', 'Observing Nature'), ('Queenstown', 'Hiking'), ('Milford Sound', 'Sightseeing Flights'), ('Stewart Island', 'Sightseeing Flights'), ('Te Anau', 'Hiking'), ('Invercargill', 'Observing Nature'), ('Catlins', 'Observing Nature'), ('Queenstown', 'Skiing'), ('Queenstown', 'Sightseeing Flights'), ('Te Anau', 'Sightseeing Flights'), ('Queenstown', 'Parachute Gliding'), ('Wanaka', 'Hiking'), (

## Part 1. Basic operations

This section introduces two main operations in FCA: the extent and the intent (both are often represented with $\cdot'$).

With these two basic operations (and some minor adjustments), we can already mine concepts and implications. Although such bruteforce algorithms can take a lot of time of the big data.

Recall that a Formal Context is a triplet $(G, M, I)$ of the set of objects $G$, the set of attributes $M$, and the incidence relation $I \subseteq G \times M$ between objects and attributes.

Extent of description $B$ is the (maximal) set of objects described by $B$:
$$\mathtt{extent}(B) = B' = \{g \in G \mid \forall m \in M\ (g, m) \in I \}$$

In [6]:
def extent(description: set[str], context: FormalContext) -> set[str]:
  # TODO: Implement the extent operation
  # HINT: Pythonic set-comprehension looks A LOT like the mathematical set-builder notation above
  return {g for g in context.objects if all((g, m) in context.relations for m in description)}

extent({'Jet Boating', 'Wildwater Rafting'}, context)

{'Queenstown', 'Wanaka'}

Intent of a subset of objects $A$ is the (maximal) set of attributes that describe $A$:
$$\mathtt{intent}(A) = A' = \{m \in M \mid \forall g \in A, (g,m) \in I \}$$

In [7]:
def intent(objects: set[str], context: FormalContext) -> set[str]:
  # TODO: Implement the intent operation
  return {m for m in context.attributes if all((g, m) in context.relations for g in objects)}

intent({'Queenstown', 'Wanaka'}, context)

{'Bungee Jumping',
 'Hiking',
 'Jet Boating',
 'Parachute Gliding',
 'Sightseeing Flights',
 'Skiing',
 'Wildwater Rafting'}

Let us import the `powerset` function from `caspailleur` package in order to iterate through the powerset of a set (i.e. iterate through all the subsets of some given set)

In [8]:
from caspailleur.base_functions import powerset

list(powerset({'a', 'b', 'c'}))

[frozenset(),
 frozenset({'a'}),
 frozenset({'b'}),
 frozenset({'c'}),
 frozenset({'a', 'b'}),
 frozenset({'a', 'c'}),
 frozenset({'b', 'c'}),
 frozenset({'a', 'b', 'c'})]

Here we implement the bruteforce algorithm for mining formal concepts: it goes through all the (exponential amount of) possible descriptions, and computes their extents and intents.

In [9]:
class FormalConcept(NamedTuple):
  extent: set[str]  # all objects covered by the concept
  intent: set[str]  # all attributes covered by the concept
  support: float = None  # the number of objects in the extent (optional)

  def __hash__(self) -> int:
    return hash((frozenset(self.extent), frozenset(self.intent)))

In [10]:
def mine_concepts_bruteforce(context: FormalContext) -> set[FormalConcept]:
  concepts = set()
  for description in powerset(context.attributes):
    # ToDo: Finish the Bruteforce algorithm for mining concepts.
    # Compute the extent and the intent of a given description and form them into a FormalConcept
    extent_ = extent(description, context)
    intent_ = intent(extent_, context)
    concepts.add(FormalConcept(extent_, intent_, len(extent_)/len(context.objects)))
  return concepts

In [11]:
%%time
concepts = mine_concepts_bruteforce(context)
pd.DataFrame(concepts).sort_values('support', ascending=False)

CPU times: user 12.1 ms, sys: 73 µs, total: 12.2 ms
Wall time: 24.4 ms


,extent,intent,support
1,"{MT. Aspiring NP, Dunedin, Stewart Island, Que...",{Hiking},1.000000
2,"{MT. Aspiring NP, Dunedin, Stewart Island, Te ...","{Observing Nature, Hiking}",0.846154
0,"{MT. Aspiring NP, Dunedin, Stewart Island, Que...","{Sightseeing Flights, Hiking}",0.692308
6,"{MT. Aspiring NP, Dunedin, Stewart Island, Te ...","{Sightseeing Flights, Observing Nature, Hiking}",0.538462
5,"{Queenstown, Te Anau, Wanaka}","{Sightseeing Flights, Hiking, Jet Boating}",0.230769
4,"{Queenstown, Wanaka}","{Jet Boating, Parachute Gliding, Bungee Jumpin...",0.153846
7,{Te Anau},"{Hiking, Sightseeing Flights, Observing Nature...",0.076923
3,{},"{Jet Boating, Parachute Gliding, Bungee Jumpin...",0.000000


Bruteforce generation of implications. Again, we pass through every subset of attributes and use it to make an implication "subset_of_attributes => its_closure". And then we filter out some of the "reduntant" implications to get the Proper Premise basis of implications.

In [12]:
class Implication(NamedTuple):
  premise: set[str]
  conclusion: set[str]
  support: float = None

  def __hash__(self) -> int:
    return hash((frozenset(self.premise), frozenset(self.conclusion)))

In [13]:
def mine_implications_bruteforce(context: FormalContext) -> set[Implication]:
  implications = set()
  for description in powerset(context.attributes):
    # ToDo: Compute the `extent_` and the `intent_` of the description to form an implication "description => intent_"
    extent_ = extent(description, context)
    intent_ = intent(extent_, context)

    # And now we filter some "obvious" and "reduntant" implications,
    # so that the final set of implications would make a Proper Premise basis (aka Canonical Direct basis)

    # if implication is obvious: i.e. description => description, then we do not want to see it
    if description == intent_:
      continue

    # if implication can be expressed by combining smaller implications, then we do not want to see it
    subdescriptions = {description - {attr} for attr in description}
    subintent = set(description)
    for subdescription in subdescriptions:
      subintent |= intent(extent(subdescription, context), context)
    if subintent == intent_:
      continue

    # we remove duplicating attributes from the conclusion, so that the implication would look more concise
    conclusion = intent_ - description

    support = len(extent_)/ len(context.objects)
    implications.add(Implication(description, conclusion, support))
  return implications

In [14]:
%%time
implications = mine_implications_bruteforce(context)
pd.DataFrame(implications).sort_values('support', ascending=False)

CPU times: user 31.8 ms, sys: 1.05 ms, total: 32.9 ms
Wall time: 34 ms


,premise,conclusion,support
3,(),{Hiking},1.000000
2,(Jet Boating),"{Sightseeing Flights, Hiking}",0.230769
1,(Parachute Gliding),"{Jet Boating, Wildwater Rafting, Bungee Jumpin...",0.153846
0,(Skiing),"{Jet Boating, Parachute Gliding, Wildwater Raf...",0.153846
4,(Wildwater Rafting),"{Jet Boating, Parachute Gliding, Bungee Jumpin...",0.153846
5,(Bungee Jumping),"{Jet Boating, Parachute Gliding, Wildwater Raf...",0.153846


## Part 2. Mining concepts in a smart way

The bruteforce way of iterating through all subsets of attributes might be very expensive when the data is big. So it is better to use some advanced algorithms. For example, NextClosure.

In [15]:
def is_lectically_smaller(description_left: set[str], description_right: set[str], index: int, attributes_list: list[str]) -> bool:
  # ToDo: Implement lectic order comparison for the next closure algorithm
  # The formula can be found on Slide 63 of the class where "description_left" is reffered to as "A" and "description_right" is reffered to as "B".
  # Note that the "index" here is a number, and not an attributes (as on the slides)
  if not (attributes_list[index] in description_right - description_left):
    return False
  return all((attributes_list[j] in description_left)==(attributes_list[j] in description_right)  for j in range(index))


def next_closure(intent_: set[str], attributes_list: list[str], context: FormalContext) -> frozenset[str]:
  # ToDo: Implement the NextClosure algorithm from Slide 65 of the class
  for i in reversed(range(len(attributes_list))):
    intent_before_i = {m for m in intent_ if attributes_list.index(m) < i}
    intent_plus_i = intent_before_i | {attributes_list[i]}
    closure_candidate = intent(extent(intent_plus_i, context), context)
    if is_lectically_smaller(intent_, closure_candidate, i, attributes_list):
      return closure_candidate
  return context.attributes

In [16]:
def all_closure(context: FormalContext) -> set[FormalConcept]:
  # AllClosure function. It is quite simple, so there is nothing to implement.
  concepts = set()

  attributes_list = sorted(context.attributes)
  intent_ = intent(extent(set(), context), context)
  while True:
    extent_ = extent(intent_, context)
    concepts.add(FormalConcept(extent_, intent_, len(extent_)/len(context.objects)))

    if intent_ == context.attributes:
      break

    intent_ = next_closure(intent_, attributes_list, context)

  return concepts

In [17]:
%%time
concepts = all_closure(context)
pd.DataFrame(concepts).sort_values('support', ascending=False)

CPU times: user 3.18 ms, sys: 331 µs, total: 3.52 ms
Wall time: 4.56 ms


,extent,intent,support
1,"{MT. Aspiring NP, Dunedin, Stewart Island, Que...",{Hiking},1.000000
2,"{MT. Aspiring NP, Dunedin, Stewart Island, Te ...","{Observing Nature, Hiking}",0.846154
0,"{MT. Aspiring NP, Dunedin, Stewart Island, Que...","{Sightseeing Flights, Hiking}",0.692308
5,"{MT. Aspiring NP, Dunedin, Stewart Island, Te ...","{Sightseeing Flights, Observing Nature, Hiking}",0.538462
6,"{Queenstown, Te Anau, Wanaka}","{Sightseeing Flights, Hiking, Jet Boating}",0.230769
4,"{Queenstown, Wanaka}","{Jet Boating, Parachute Gliding, Bungee Jumpin...",0.153846
7,{Te Anau},"{Hiking, Sightseeing Flights, Observing Nature...",0.076923
3,{},"{Jet Boating, Parachute Gliding, Bungee Jumpin...",0.000000


Make sure that the bruteforce and the smart way give the same output

In [18]:
assert all_closure(context) == mine_concepts_bruteforce(context)

In [19]:
%timeit mine_concepts_bruteforce(context)
%timeit all_closure(context)

12.9 ms ± 4.36 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.4 ms ± 1.22 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Part 3. Mining implications in a smart(er) way

In this Part we mine Proper Premise basis of implications in a smarter way.

Recall that a **proper premise** is defined as follows:
$$ P \subseteq M \text{ is a proper premise} \iff P \cup \bigcup_{m \in P} (P \setminus \{m\})'' \neq P''.$$

The mathematical definition defines an important property:
$$P \cup \bigcup_{m \in P} (P \setminus \{m\})'' \neq P'' \implies (P \setminus \{m\})'' \neq P'', \forall m \in P,\quad \text{ i.e. } P \text{ is a minimal generator}.$$
And **minimal generator** is a description, its every subset of attributes describes more objects. So, you cannot make a minimal generator smaller but still describe the same objects.

That is, if all proper premises are minimal generators, then we can find all minimal generators and filter out the ones that are not proper premises.

Below is one of a simple-yet-efficient algorithms for mining minimal generators called Carpathia-G.

The algorithm uses one property of a minimal generator: $$\text{every subset of a minimal generator is a minimal generator}.$$

That is, if there is some description s.t. one of its subsets is not a minimal generator, then the description itself is not a minimal generator.

In [20]:
def carpathia_g(context: FormalContext) -> list[set[str]]:
  mingens: set[frozenset[str]] = set()
  attributes_list = list(context.attributes)

  # The algorithm goes through various subsets of attributes, starting from the smallest one (the empty subset) and incrementally going to more and more complex descriptions
  queue = [frozenset()]
  while queue:
    description = queue.pop(0)
    subdescriptions = {description - {attr} for attr in description}
    # ToDo: Implement Test #1: if some subdescriptions are not in `mingens` set, the `description` is not a minimal generator
    if not all(subdescription in mingens for subdescription in subdescriptions):
      continue  # not a minimal generator because not all subsets are minimal generators

    # ToDo: Implement Test #2: if some subdescription is describes the same extent, then `description` is not a minimal generator
    extent_ = extent(description, context)
    if any(extent(subdescription, context) == extent_ for subdescription in subdescriptions):
      continue  # description itself is not a minimal generator
    mingens.add(description)

    # If the two tests were successfull, then we can try to add one more attributes to the description
    max_idx = max(attributes_list.index(attr) for attr in description) if description else -1
    next_descriptions = [description | {m} for m in attributes_list[max_idx+1:]]
    queue.extend(next_descriptions)

  return [frozenset(mingen) for mingen in mingens]

We know how to iterate minimal generators (using Carpathia-G), now we just should identify the minimal generators that are also proper premises. You can do that by following the definition of the proper premise above.

In [21]:
def is_proper_premise(min_generator: frozenset[str], context: FormalContext) -> bool:
  extent_ = extent(min_generator, context)
  intent_ = intent(extent_, context)

  if min_generator == intent_: # mingen is a min.generator and gives not clearly obvious implication
    return False

  subintent = set(min_generator)
  subdescriptions = {min_generator - {attr} for attr in min_generator}
  for subdescription in subdescriptions:
    subintent |= intent(extent(subdescription, context), context)
  return subintent != intent_

In [22]:
def mine_implications_smart(context: FormalContext) -> set[Implication]:
  implications: set[Implication] = set()

  attributes_list = list(context.attributes)

  # the algorithm is simple: go through every minimal generator, if it is a proper premise, then make it an Implication
  for mingen in carpathia_g(context):
    if is_proper_premise(mingen, context):
      extent_ = extent(mingen, context)
      conclusion = intent(extent_, context) - mingen
      support = len(extent_) / len(context.objects)
      implications.add(Implication(set(mingen), conclusion, support))

  return implications

Make sure that the bruteforce and the smart way give the smae output

In [23]:
assert mine_implications_smart(context) == mine_implications_bruteforce(context)

In [24]:
%timeit mine_implications_bruteforce(context)
%timeit mine_implications_smart(context)

47.4 ms ± 4.34 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
3.15 ms ± 204 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Part 4. Association rules

Implications can often be too "strict" to use them on real data. One of the solutions is to mine **association rules** that are just like the implications rules but with varying confidence.

In [25]:
class AssociationRule(NamedTuple):
  premise: set[str]
  conclusion: set[str]
  confidence: float = None
  support: float = None

  def __hash__(self):
    return hash((frozenset(self.premise), frozenset(self.conclusion)))

One of the classical way to compute the basis of association rules is to use so-called Luxenburger basis. It finds all pairs of neighbouring concepts in a concept lattice and makes an association rule that the intent of the greater concept associates to the intent of the smaller concept.

In [28]:
def luxenburger_basis(context: FormalContext) -> set[AssociationRule]:
  concepts = all_closure(context)

  associations = set()

  for concept in concepts:
    subconcepts = {other for other in concepts if other.extent < concept.extent}
    top_subconcepts = {
        subconcept for subconcept in subconcepts
        if not any(subconcept.extent < higher_sub.extent for higher_sub in subconcepts)
    }

    for subconcept in top_subconcepts:
      confidence = len(subconcept.extent)/len(concept.extent) if subconcept.extent else 1
      support = len(subconcept.extent) / len(context.objects)

      conclusion = subconcept.intent - concept.intent
      associations.add(AssociationRule(concept.intent, conclusion, confidence, support))
  return associations

In [29]:
pd.DataFrame(luxenburger_basis(context))

,premise,conclusion,confidence,support
0,"{Hiking, Sightseeing Flights, Observing Nature...","{Wildwater Rafting, Parachute Gliding, Skiing,...",1.000000,0.000000
1,{Hiking},{Observing Nature},0.846154,0.846154
2,"{Sightseeing Flights, Hiking, Jet Boating}",{Observing Nature},0.333333,0.076923
3,{Hiking},{Sightseeing Flights},0.692308,0.692308
4,"{Sightseeing Flights, Hiking, Jet Boating}","{Wildwater Rafting, Parachute Gliding, Skiing,...",0.666667,0.153846
5,"{Sightseeing Flights, Hiking}",{Jet Boating},0.333333,0.230769
6,"{Sightseeing Flights, Observing Nature, Hiking}",{Jet Boating},0.142857,0.076923
7,"{Observing Nature, Hiking}",{Sightseeing Flights},0.636364,0.538462
8,"{Sightseeing Flights, Hiking}",{Observing Nature},0.777778,0.538462
9,"{Jet Boating, Parachute Gliding, Bungee Jumpin...",{Observing Nature},1.000000,0.000000


## Part 5. The Real data

Let us try to run FCA algorithms on some real data, e.g. the Titanic one.
All the code here is already written but you are invited to generate new binary attributes on this complex data.

There are two important questions: 1) what are some interesting dependancies on the Titanic data, and 2) at what point the running time of FCA algorithm will exponentially explore.

In [30]:
titanic_df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv', index_col=0)
titanic_df.index = 'Passenger '+titanic_df.index.astype(str)
titanic_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
Passenger 1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
Passenger 2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
Passenger 3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
Passenger 4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
Passenger 5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [31]:
# Here are some of possible binary attributes. You can always propose your own attributes.
titanic_bin = {
    'Survived': titanic_df['Survived'] == 1,
    'FirstClass': titanic_df['Pclass'] == 1,
    'French': titanic_df['Embarked'] == 'C',  # 'C' means 'Cherbourg'
    'Adult': titanic_df['Age'] >= 18,
    'Child': titanic_df['Age'] < 18,
    'Male': titanic_df['Sex'] == 'male',
    'Female': titanic_df['Sex'] == 'female',
}
titanic_bin = pd.DataFrame(titanic_bin)
titanic_bin.head()

,Survived,FirstClass,French,Adult,Child,Male,Female
PassengerId,,,,,,,
Passenger 1,False,False,False,True,False,True,False
Passenger 2,True,True,True,True,False,False,True
Passenger 3,True,False,False,True,False,False,True
Passenger 4,True,True,False,True,False,False,True
Passenger 5,False,False,False,True,False,True,False


In [32]:
titanic_context = pandas_to_sets(titanic_bin)

In [33]:
assert mine_concepts_bruteforce(titanic_context) == all_closure(titanic_context)
%timeit mine_concepts_bruteforce(titanic_context)
%timeit all_closure(titanic_context)

666 ms ± 199 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
970 ms ± 507 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Note that sometimes advanced algorithms can work slower than the bruteforce ones: that happens when the data is too simple.

In [34]:
concepts = all_closure(titanic_context)
pd.DataFrame(concepts).sort_values('support', ascending=False)

,extent,intent,support
31,"{Passenger 533, Passenger 149, Passenger 534, ...",{},1.000000
43,"{Passenger 149, Passenger 1, Passenger 563, Pa...",{Adult},0.674523
58,"{Passenger 533, Passenger 149, Passenger 1, Pa...",{Male},0.647587
12,"{Passenger 149, Passenger 1, Passenger 563, Pa...","{Adult, Male}",0.443322
0,"{Passenger 534, Passenger 10, Passenger 756, P...",{Survived},0.383838
...,...,...,...
7,"{Passenger 306, Passenger 446, Passenger 803, ...","{FirstClass, Survived, Child, Male}",0.004489
22,"{Passenger 308, Passenger 551, Passenger 330}","{FirstClass, Survived, Child, French}",0.003367
1,"{Passenger 308, Passenger 330}","{Female, FirstClass, Child, French, Survived}",0.002245
13,{Passenger 551},"{FirstClass, Child, Male, French, Survived}",0.001122


In [35]:
assert mine_implications_smart(titanic_context) == mine_implications_bruteforce(titanic_context)
%timeit mine_implications_bruteforce(titanic_context)
%timeit mine_implications_smart(titanic_context)

1.01 s ± 229 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
497 ms ± 76 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
implications = mine_implications_smart(titanic_context)
pd.DataFrame(implications).sort_values('support', ascending=False)

,premise,conclusion,support
2,"{FirstClass, Child, Male}",{Survived},0.004489
3,"{FirstClass, Child, French}",{Survived},0.003367
1,"{Female, Male}","{FirstClass, Child, Adult, French, Survived}",0.000000
0,"{Child, Adult}","{Female, FirstClass, Male, French, Survived}",0.000000


In [37]:
associations = luxenburger_basis(titanic_context)
association_df = pd.DataFrame(associations).sort_values(['support', 'confidence'], ascending=False).reset_index(drop=True)
association_df = association_df[(association_df['confidence']>0.5)&(association_df['support']>=0.1)]
association_df

,premise,conclusion,confidence,support
0,{},{Adult},0.674523,0.674523
1,{},{Male},0.647587,0.647587
2,{Male},{Adult},0.684575,0.443322
3,{Adult},{Male},0.657238,0.443322
6,{Female},{Survived},0.742038,0.261504
7,{Survived},{Female},0.681287,0.261504
8,{Survived},{Adult},0.669591,0.257015
11,{Female},{Adult},0.656051,0.231201
13,{FirstClass},{Adult},0.805556,0.195286
16,"{Female, Adult}",{Survived},0.771845,0.178451


## Part 6. Integration with GraphRAG and Ontologies

And here is some hint on what will happen during the second half of the course and during the project work.

Except that we will learn how to find concepts, implications, and associations on complex data (i.e. numerical, categorical, graphs, etc.), so the binarisation step will become obsolete.

### Part 6.1. Create ontology

In [38]:
!pip install --quiet git+https://github.com/smartFCA/OntologyPailleur

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


We should change the concepts representation a little bit. In `OntologyPailleur` package every formal concept is a triple of (extent, intent, minimal_generators).


In [39]:
mingens_per_closure = {frozenset(implication.premise | implication.conclusion): [] for implication in implications}
for implication in implications:
  closure = frozenset(implication.premise | implication.conclusion)
  mingens_per_closure[closure].append(implication.premise)

In [40]:
from ontopailleur import formal_ontology_constructor as foc
concepts_to_save = [foc.FormalConcept(extent(intent, titanic_context), intent, mingens) for intent, mingens in mingens_per_closure.items()]

We use `rdflib` to create knowledge graphs in Turtle format (.ttl) within Python.

Every node of a knowledge graph (including the ones representing objects $G$ and attributes $M$), should be represented with some `rdflib.URIRef(node_name)`. The name of the node should start with a prefix (that can be done via `rdflib.Namespace`), and should contain neither empty spaces nor special symbols.

The code below compute automatic URIRef's for each object and attribute, but you might need to change the code if you have some particular names of objects and attributes.

In [41]:
import rdflib
NSpace = rdflib.Namespace('http://idmc.univ-lorraine.fr/td_fca#')
obj_to_refs = {obj: rdflib.URIRef(NSpace[obj.replace(' ', '_').capitalize()]) for obj in titanic_context.objects}
attr_to_refs = {attr: rdflib.URIRef(NSpace[attr.replace(' ', '_').capitalize()]) for attr in titanic_context.attributes}

attr_to_refs

{'Female': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Female'),
 'FirstClass': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Firstclass'),
 'Child': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Child'),
 'Adult': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Adult'),
 'Male': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Male'),
 'French': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#French'),
 'Survived': rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Survived')}

In [42]:
kgraph = foc.construct_ontology(NSpace, prefix='tdfca', objects_to_refs=obj_to_refs, attributes_to_refs=attr_to_refs, incidence_relation=titanic_context.relations, concepts=concepts_to_save)
print("First 5 RDF triplets of the knowledge graph")
list(kgraph.triples((None, None, None)))[:5]

First 5 RDF triplets of the knowledge graph


[(rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_374'),
  rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'),
  rdflib.term.Literal('Passenger 374')),
 (rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_328'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
  rdflib.term.URIRef('http://www.w3.org/2002/07/owl#NamedIndividual')),
 (rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_627'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
  rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Adult')),
 (rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_90'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
  rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Male')),
 (rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_820'),
  rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'),
  rdflib.term.Literal('

In [43]:
kgraph.serialize('kgraph.ttl')

<Graph identifier=N9950f20b25464330b896a051fadb2e94 (<class 'rdflib.graph.Graph'>)>

You can download the saved graph and open it in Protégé system as an ontology.

### Part 6.2 Running GraphRAG

Now we want to put the obtained Knowledge Graph into a GraphRAG pipeline. To do so, we should download some more packages and neural networks.

In [44]:
!pip install --quiet networkx matplotlib torch transformers sentence-transformers lmdb -q
!pip install --quiet tf_keras

In [45]:
from ontopailleur import graphrag

EMBEDDING_MODEL_NAME: str = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME: str = 'Qwen/Qwen2-1.5B-Instruct'  # 'Qwen/Qwen2-0.5B-Instruct'
LMDB_PATH: str = './embeddings_local'
GRAPH_PATH = 'kgraph.ttl'

In [46]:
embedding_model = graphrag.compute_embeddings(GRAPH_PATH, EMBEDDING_MODEL_NAME, LMDB_PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Compute embeddings:   0%|          | 0/1807 [00:00<?, ?it/s]

LMDB database created: ./embeddings_local


In [47]:
grag = graphrag.GraphRAG(GRAPH_PATH, LMDB_PATH, embedding_model)

Graph loaded!
1821 nodes
4188 edges


In [48]:
tokenizer, model = graphrag.initialise_llm(LLM_MODEL_NAME)

Loading Qwen/Qwen2-1.5B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded!


Now we can ask use GraphRAG system for asking questions about the data.

In [49]:
graphrag.graphrag_query(grag, tokenizer, model, 'What describes Passenger 520?', hops=1)

Query: What describes Passenger 520?
Semantic search for entities ...
5 entities found
Passenger 520 -> similarity score: 0.8827335238456726)
Passenger 520 -> similarity score: 0.8827335238456726)
Passenger 525 -> similarity score: 0.8005335927009583)
Graph expansion 1-hop ...
Subgraph: 11 nodes, 12 edges
Generating answer...
Answer:
Passenger 53 is male


{'question': 'What describes Passenger 520?',
 'entities': [rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_520'),
  rdflib.term.Literal('Passenger 520'),
  rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_525'),
  rdflib.term.Literal('Passenger 525'),
  rdflib.term.URIRef('http://idmc.univ-lorraine.fr/td_fca#Passenger_475')],
 'scores': [np.float32(0.8827335),
  np.float32(0.8827335),
  np.float32(0.8005336),
  np.float32(0.8005336),
  np.float32(0.783711)],
 'subgraph': <networkx.classes.digraph.DiGraph at 0x7ef38bcdb9e0>,
 'facts': ['Passenger 475 Http://Www.W3.Org/2000/01/Rdf-Schema#Label Passenger 475',
  'Passenger 475 Http://Www.W3.Org/1999/02/22-Rdf-Syntax-Ns#Type Female',
  'Passenger 475 Http://Www.W3.Org/1999/02/22-Rdf-Syntax-Ns#Type Http://Www.W3.Org/2002/07/Owl#Namedindividual',
  'Passenger 475 Http://Www.W3.Org/1999/02/22-Rdf-Syntax-Ns#Type Adult',
  'Passenger 525 Http://Www.W3.Org/2000/01/Rdf-Schema#Label Passenger 525',
  'Passenger 